## STEP 0: Improt Training data 


In [29]:
from collections import defaultdict
import pandas as pd
import numpy as np
import pickle



In [30]:
df = pd.read_csv('split_cleaned_dataset.csv', sep=';')
print("First 5 rows:")
print(df.head())
print("\nSplit distribution:")
print(df['split'].value_counts())

First 5 rows:
   Rank              Title                     Artists        Date  \
0     1    Ella Baila Sola  Eslabon Armado, Peso Pluma  2023-05-29   
1     2     WHERE SHE GOES                   Bad Bunny  2023-05-29   
2     3    La Bebe - Remix       Yng Lvcas, Peso Pluma  2023-05-29   
3     4  Cupid - Twin Ver.                 FIFTY FIFTY  2023-05-29   
4     5          un x100to   Grupo Frontera, Bad Bunny  2023-05-29   

   Danceability  Energy  Loudness  Speechiness  Acousticness  \
0         0.668   0.758    -5.176        0.033         0.483   
1         0.652   0.800    -4.019        0.061         0.143   
2         0.812   0.479    -5.678        0.333         0.213   
3         0.783   0.592    -8.332        0.033         0.435   
4         0.569   0.724    -4.076        0.047         0.228   

   Instrumentalness  ...  # of Nationality  Nationality      Continent  \
0             0.000  ...     Nationality 1       Mexico  Latin-America   
1             0.629  ...     Nat

In [31]:
# Filter only training data
raw_train = df[df['split'] == 'train'].copy()

print(f"Total rows: {len(df)}")
print(f"Training rows: {len(raw_train)}")

# Show sample of training data
print("\nTraining data sample:")
print(raw_train.head())
print(f"\nColumns: {raw_train.columns.tolist()}")


Total rows: 467061
Training rows: 345628

Training data sample:
    Rank                             Title                         Artists  \
15    18  See You Again (feat. Kali Uchis)  Tyler, The Creator, Kali Uchis   
16    19                   Angels Like You                     Miley Cyrus   
22    25                       Die For You                      The Weeknd   
27    30                      Cruel Summer                    Taylor Swift   
37    43                           Starboy           The Weeknd, Daft Punk   

          Date  Danceability  Energy  Loudness  Speechiness  Acousticness  \
15  2023-05-29         0.558   0.559    -9.222        0.096         0.371   
16  2023-05-29         0.672   0.642    -4.035        0.031         0.098   
22  2023-05-29         0.586   0.525    -7.163        0.062         0.111   
27  2023-05-29         0.552   0.702    -5.707        0.157         0.117   
37  2023-05-29         0.679   0.587    -7.015        0.276         0.141   

    

## STEP 1: Calculate Artist Features
Purpose: Get statistics for each unique artist (primary + collaborators)

1A: Extract All Artist Appearances

In [32]:
artist_appearances = defaultdict(list)

for idx, row in raw_train.iterrows():
    song_id = row['id']
    title = row['Title']
    artists_str = row['Artists']
    date = row['Date']
    rank = row['Rank']

    artists_list = [artist.strip() for artist in artists_str.split(',')]
    for position, artist_name in enumerate(artists_list):
        is_primary = (position == 0)
        
        appearance = {
            'song_id': song_id,
            'title': title,
            'date': date,
            'rank': rank,
            'is_primary': is_primary,
            'role': 'primary' if is_primary else 'collaborator',
            'artist_position': position
        }
        
        artist_appearances[artist_name].append(appearance)

print(artist_appearances['Ed Sheeran'][:2])  # Sample output for an artist
# Convert to DataFrame
all_appearances = []
for artist_name, appearances in artist_appearances.items():
    for appearance in appearances:
        all_appearances.append({
            'artist_name': artist_name,
            **appearance
        })

artist_appearances_df = pd.DataFrame(all_appearances)
artist_appearances_df['date'] = pd.to_datetime(artist_appearances_df['date'])

print(f"\nExtracted {len(artist_appearances_df)} artist appearances from {len(raw_train)} training rows")
print(f"Unique artists: {len(artist_appearances)}")
print("\nFirst 5 rows:")
print(artist_appearances_df.head())
print(f"\nData types:\n{artist_appearances_df.dtypes}")

[{'song_id': '0tgVpDi06FyKpA1z0VMD4v', 'title': 'Perfect', 'date': '2023-05-29', 'rank': 94, 'is_primary': True, 'role': 'primary', 'artist_position': 0}, {'song_id': '7qiZfU4dY1lWllzX7mPBI3', 'title': 'Shape of You', 'date': '2023-05-29', 'rank': 164, 'is_primary': True, 'role': 'primary', 'artist_position': 0}]

Extracted 474470 artist appearances from 345628 training rows
Unique artists: 1469

First 5 rows:
  artist_name                 song_id                             title  \
0       Tyler  7KA4W4McWYRpgf0fWsJZWB  See You Again (feat. Kali Uchis)   
1       Tyler  7KA4W4McWYRpgf0fWsJZWB  See You Again (feat. Kali Uchis)   
2       Tyler  7KA4W4McWYRpgf0fWsJZWB  See You Again (feat. Kali Uchis)   
3       Tyler  7KA4W4McWYRpgf0fWsJZWB  See You Again (feat. Kali Uchis)   
4       Tyler  7KA4W4McWYRpgf0fWsJZWB  See You Again (feat. Kali Uchis)   

        date  rank  is_primary     role  artist_position  
0 2023-05-29    18        True  primary                0  
1 2023-05-28    1

1B: Calculate Artist Statistics

In [33]:
 #Group by artist and calculate statistics
primary_only = artist_appearances_df[artist_appearances_df['is_primary'] == True]
artist_stats = primary_only.groupby('artist_name').agg({
    'song_id': 'nunique',      # Count unique songs
    'rank': ['mean', 'min']     # Average and best rank
}).reset_index()

# Flatten column names
artist_stats.columns = ['artist_name', 'song_count', 'avg_rank', 'best_rank']

# Round avg_rank to 2 decimal places for readability
artist_stats['avg_rank'] = artist_stats['avg_rank'].round(2)

# Sort by song_count descending to see most prolific artists first
artist_stats = artist_stats.sort_values('song_count', ascending=False)

print(f"\nTotal unique artists: {len(artist_stats):,}")
print("\nArtist Statistics Summary:")
print("="*80)
print(artist_stats.describe())

print("\n" + "="*80)
print("Top 20 Artists by Song Count:")
print("="*80)
print(artist_stats.head(20).to_string(index=False))

print("\n" + "="*80)
print("Top 20 Artists by Best Rank (Best Performers):")
print("="*80)
top_performers = artist_stats.nsmallest(20, 'best_rank')
print(top_performers[['artist_name', 'song_count', 'avg_rank', 'best_rank']].to_string(index=False))

print("\n" + "="*80)
print("Examples from Spec:")
print("="*80)

# Check Bad Bunny
if 'Bad Bunny' in artist_stats['artist_name'].values:
    bb = artist_stats[artist_stats['artist_name'] == 'Bad Bunny'].iloc[0]
    print(f"\nBad Bunny:")
    print(f"  - Unique songs: {bb['song_count']}")
    print(f"  - Average rank: {bb['avg_rank']}")
    print(f"  - Best rank: {bb['best_rank']}")

# Check Peso Pluma
if 'Peso Pluma' in artist_stats['artist_name'].values:
    pp = artist_stats[artist_stats['artist_name'] == 'Peso Pluma'].iloc[0]
    print(f"\nPeso Pluma:")
    print(f"  - Unique songs: {pp['song_count']}")
    print(f"  - Average rank: {pp['avg_rank']}")
    print(f"  - Best rank: {pp['best_rank']}")

# Check J Balvin
if 'J Balvin' in artist_stats['artist_name'].values:
    jb = artist_stats[artist_stats['artist_name'] == 'J Balvin'].iloc[0]
    print(f"\nJ Balvin:")
    print(f"  - Unique songs: {jb['song_count']}")
    print(f"  - Average rank: {jb['avg_rank']}")
    print(f"  - Best rank: {jb['best_rank']}")

print("\n" + "="*80)
print("Distribution of Song Counts:")
print("="*80)
song_count_dist = artist_stats['song_count'].value_counts().sort_index()
print("\nArtists by number of unique songs:")
for count, num_artists in song_count_dist.head(20).items():
    print(f"  {count} song(s): {num_artists:,} artists")

print(f"\nDataFrame shape: {artist_stats.shape}")
print(f"Columns: {artist_stats.columns.tolist()}")


Total unique artists: 1,030

Artist Statistics Summary:
        song_count     avg_rank    best_rank
count  1030.000000  1030.000000  1030.000000
mean      5.984466   134.422961    79.933981
std      11.297538    35.555800    60.095369
min       1.000000    31.020000     1.000000
25%       1.000000   105.630000    25.000000
50%       2.000000   134.775000    71.500000
75%       5.000000   163.305000   127.000000
max     114.000000   200.000000   200.000000

Top 20 Artists by Song Count:
     artist_name  song_count  avg_rank  best_rank
           Drake         114     98.17          1
          Future          99    103.37          3
    Taylor Swift          95    104.19          1
             BTS          87     90.00          1
          Eminem          76    124.62          1
      Juice WRLD          75    100.53          2
           Logic          75     95.62          2
     Post Malone          74     88.94          1
   Ariana Grande          72     87.21          1
    XXX

1C: Convert to Dictionary (Lookup)

In [34]:
artist_stats_lookup = {}

for _, row in artist_stats.iterrows():
    artist_stats_lookup[row['artist_name']] = {
        'song_count': int(row['song_count']),
        'avg_rank': float(row['avg_rank']),
        'best_rank': int(row['best_rank'])
    }

print(f"\nTotal artists in lookup dictionary: {len(artist_stats_lookup):,}")

# Verify with examples from spec
print("\n" + "="*80)
print("Examples from Spec:")
print("="*80)

example_artists = ['Bad Bunny', 'Peso Pluma', 'J Balvin']

for artist in example_artists:
    if artist in artist_stats_lookup:
        stats = artist_stats_lookup[artist]
        print(f"\n{artist}:")
        print(f"  - song_count: {stats['song_count']}")
        print(f"  - avg_rank: {stats['avg_rank']}")
        print(f"  - best_rank: {stats['best_rank']}")
    else:
        print(f"\n{artist}: Not found in dataset")

# Show a few more examples
print("\n" + "="*80)
print("Sample Lookup Entries (First 10):")
print("="*80)

for i, (artist_name, stats) in enumerate(list(artist_stats_lookup.items())[:10], 1):
    print(f"\n{i}. '{artist_name}': {{")
    print(f"     'song_count': {stats['song_count']},")
    print(f"     'avg_rank': {stats['avg_rank']},")
    print(f"     'best_rank': {stats['best_rank']}")
    print(f"   }}")

# Demonstrate fast lookup
print("\n" + "="*80)
print("Testing Fast Lookup:")
print("="*80)

test_artist = 'Ed Sheeran'
if test_artist in artist_stats_lookup:
    print(f"\nLookup result for '{test_artist}':")
    print(artist_stats_lookup[test_artist])
else:
    print(f"\n'{test_artist}' not found")

# Show dictionary structure
print("\n" + "="*80)
print("Dictionary Structure:")
print("="*80)
print(f"Type: {type(artist_stats_lookup)}")
print(f"Number of keys: {len(artist_stats_lookup)}")
print(f"Memory efficient: O(1) lookup time")

# Verify data types
sample_artist = list(artist_stats_lookup.keys())[0]
sample_stats = artist_stats_lookup[sample_artist]
print("\nValue data types:")
print(f"  - song_count: {type(sample_stats['song_count']).__name__}")
print(f"  - avg_rank: {type(sample_stats['avg_rank']).__name__}")
print(f"  - best_rank: {type(sample_stats['best_rank']).__name__}")

print("\n" + "="*80)
print("✓ Dictionary conversion complete!")
print("="*80)


Total artists in lookup dictionary: 1,030

Examples from Spec:

Bad Bunny:
  - song_count: 63
  - avg_rank: 83.85
  - best_rank: 1

Peso Pluma: Not found in dataset

J Balvin:
  - song_count: 40
  - avg_rank: 97.88
  - best_rank: 1

Sample Lookup Entries (First 10):

1. 'Drake': {
     'song_count': 114,
     'avg_rank': 98.17,
     'best_rank': 1
   }

2. 'Future': {
     'song_count': 99,
     'avg_rank': 103.37,
     'best_rank': 3
   }

3. 'Taylor Swift': {
     'song_count': 95,
     'avg_rank': 104.19,
     'best_rank': 1
   }

4. 'BTS': {
     'song_count': 87,
     'avg_rank': 90.0,
     'best_rank': 1
   }

5. 'Eminem': {
     'song_count': 76,
     'avg_rank': 124.62,
     'best_rank': 1
   }

6. 'Juice WRLD': {
     'song_count': 75,
     'avg_rank': 100.53,
     'best_rank': 2
   }

7. 'Logic': {
     'song_count': 75,
     'avg_rank': 95.62,
     'best_rank': 2
   }

8. 'Post Malone': {
     'song_count': 74,
     'avg_rank': 88.94,
     'best_rank': 1
   }

9. 'Ariana Gr

## STEP 2: Calculate Collaboration Features
Purpose: Get statistics about collaborators specifically

2A: Extract All Collaborators

In [35]:
collaborator_data = []

# Loop through training data
for idx, row in raw_train.iterrows():
    song_id = row['id']
    artists_str = row['Artists']
    date = row['Date']
    rank = row['Rank']
    
    # Parse artists - split by comma
    artists_list = [artist.strip() for artist in artists_str.split(',')]
    
    # First artist is primary, rest are collaborators
    primary_artist = artists_list[0]
    collaborators = artists_list[1:]  # Everything after first artist
    
    # Record each collaborator appearance
    if len(collaborators) > 0:
        for collaborator_name in collaborators:
            collaborator_data.append({
                'collaborator': collaborator_name,
                'song_id': song_id,
                'rank': rank,
                'date': date
            })

# Convert to DataFrame
collaborator_df = pd.DataFrame(collaborator_data)
collaborator_df['date'] = pd.to_datetime(collaborator_df['date'])

print(f"\nTotal collaborator appearances extracted: {len(collaborator_df):,}")
print(f"Unique collaborators: {collaborator_df['collaborator'].nunique():,}")
print(f"Unique songs with collaborators: {collaborator_df['song_id'].nunique():,}")

print("\n" + "="*80)
print("Sample Data (First 20 rows):")
print("="*80)
print(collaborator_df.head(20).to_string(index=False))

print("\n" + "="*80)
print("Example: Bad Bunny as Collaborator (First 10 appearances):")
print("="*80)
if 'Bad Bunny' in collaborator_df['collaborator'].values:
    bb_collab = collaborator_df[collaborator_df['collaborator'] == 'Bad Bunny'].head(10)
    print(bb_collab.to_string(index=False))
else:
    print("Bad Bunny not found as collaborator")

print("\n" + "="*80)
print("Top 20 Most Frequent Collaborators:")
print("="*80)
collab_counts = collaborator_df['collaborator'].value_counts().head(20)
for artist, count in collab_counts.items():
    print(f"  {artist}: {count:,} appearances")

print("\n" + "="*80)
print("DataFrame Info:")
print("="*80)
print(f"Shape: {collaborator_df.shape}")
print(f"Columns: {collaborator_df.columns.tolist()}")
print(f"Date range: {collaborator_df['date'].min()} to {collaborator_df['date'].max()}")
print(f"Rank range: {collaborator_df['rank'].min()} to {collaborator_df['rank'].max()}")

print("\n✓ Collaborator extraction complete!")


Total collaborator appearances extracted: 128,842
Unique collaborators: 765
Unique songs with collaborators: 1,369

Sample Data (First 20 rows):
collaborator                song_id  rank       date
 The Creator 7KA4W4McWYRpgf0fWsJZWB    18 2023-05-29
  Kali Uchis 7KA4W4McWYRpgf0fWsJZWB    18 2023-05-29
   Daft Punk 7MXVkk9YMctZqd1Srtv4MB    43 2023-05-29
    Jay Rock 2HbKqm4o0w5wEeEFXm2sD4    63 2023-05-29
    Swae Lee 0RiRZpuVRbi7oqRdSMwhQY   100 2023-05-29
      Khalid 0u2P5u6lvoDfwTYjAADbn4   114 2023-05-29
    Coldplay 6RUKPb4LETWmmr3iAEQktW   177 2023-05-29
      Wizkid 1zi7xx7UVEFkmKfv06H8x0   198 2023-05-29
        Kyla 1zi7xx7UVEFkmKfv06H8x0   198 2023-05-29
 The Creator 7KA4W4McWYRpgf0fWsJZWB    19 2023-05-28
  Kali Uchis 7KA4W4McWYRpgf0fWsJZWB    19 2023-05-28
   Daft Punk 7MXVkk9YMctZqd1Srtv4MB    44 2023-05-28
    Jay Rock 2HbKqm4o0w5wEeEFXm2sD4    70 2023-05-28
    Swae Lee 0RiRZpuVRbi7oqRdSMwhQY   100 2023-05-28
      Khalid 0u2P5u6lvoDfwTYjAADbn4   119 2023-05-28
      

2B: Count Collaborator Frequencies

In [36]:
collaborator_frequency = collaborator_df.groupby('collaborator').size().reset_index(name='frequency')

# Sort by frequency descending
collaborator_frequency = collaborator_frequency.sort_values('frequency', ascending=False)

print(f"\nTotal unique collaborators: {len(collaborator_frequency):,}")
print(f"Total collaborator appearances: {collaborator_frequency['frequency'].sum():,}")

print("\n" + "="*80)
print("Top 20 Most Frequent Collaborators:")
print("="*80)
print(collaborator_frequency.head(20).to_string(index=False))

print("\n" + "="*80)
print("Examples from Spec:")
print("="*80)

example_collabs = ['Bad Bunny', 'J Balvin', 'Peso Pluma', 'Ozuna']

for collab in example_collabs:
    if collab in collaborator_frequency['collaborator'].values:
        freq = collaborator_frequency[collaborator_frequency['collaborator'] == collab].iloc[0]['frequency']
        print(f"\n{collab}:")
        print(f"  - Frequency: {freq:,} appearances as collaborator")
    else:
        print(f"\n{collab}: Not found as collaborator")

print("\n" + "="*80)
print("Frequency Distribution:")
print("="*80)
print(f"Mean frequency: {collaborator_frequency['frequency'].mean():.2f}")
print(f"Median frequency: {collaborator_frequency['frequency'].median():.0f}")
print(f"Min frequency: {collaborator_frequency['frequency'].min()}")
print(f"Max frequency: {collaborator_frequency['frequency'].max()}")

print("\n" + "="*80)
print("Bottom 10 (Least Frequent Collaborators):")
print("="*80)
print(collaborator_frequency.tail(10).to_string(index=False))

print("\n✓ Collaborator frequency counting complete!")


Total unique collaborators: 765
Total collaborator appearances: 128,842

Top 20 Most Frequent Collaborators:
  collaborator  frequency
      J Balvin       5352
     Bad Bunny       4778
         Ozuna       3623
        Khalid       2854
  Daddy Yankee       2811
      Anuel AA       2664
      Swae Lee       2411
       Farruko       2225
      Dua Lipa       1894
 Lenny Tavárez       1818
         Quavo       1664
     Nicky Jam       1558
     Daft Punk       1390
   Nicki Minaj       1349
          Sech       1341
      Coldplay       1301
Bradley Cooper       1288
        Halsey       1280
        Darell       1249
   Myke Towers       1230

Examples from Spec:

Bad Bunny:
  - Frequency: 4,778 appearances as collaborator

J Balvin:
  - Frequency: 5,352 appearances as collaborator

Peso Pluma: Not found as collaborator

Ozuna:
  - Frequency: 3,623 appearances as collaborator

Frequency Distribution:
Mean frequency: 168.42
Median frequency: 30
Min frequency: 1
Max frequency: 5352


2C: Identify Top 15 Collaborators

In [37]:
top_collabs = collaborator_frequency.nlargest(15, 'frequency')

print("\nTop 15 Most Frequent Collaborators:")
print("="*80)
print(top_collabs.to_string(index=False))

# Extract as list for feature engineering
top_15_collabs_list = top_collabs['collaborator'].tolist()

print("\n" + "="*80)
print("Top 15 Collaborators List (for binary features):")
print("="*80)
for i, collab in enumerate(top_15_collabs_list, 1):
    freq = top_collabs[top_collabs['collaborator'] == collab].iloc[0]['frequency']
    print(f"{i:2d}. {collab:<30} (Frequency: {freq:,})")

print("\n" + "="*80)
print("Summary:")
print("="*80)
print(f"Total collaborators identified: {len(top_15_collabs_list)}")
print(f"Total appearances covered: {top_collabs['frequency'].sum():,}")
print(f"Percentage of all collaborator appearances: {(top_collabs['frequency'].sum() / collaborator_frequency['frequency'].sum() * 100):.2f}%")

print("\n" + "="*80)
print("List stored in variable: top_15_collabs_list")
print("="*80)
print(f"Type: {type(top_15_collabs_list)}")
print(f"Length: {len(top_15_collabs_list)}")
print(f"\nFirst 5: {top_15_collabs_list[:5]}")
print(f"Last 5: {top_15_collabs_list[-5:]}")

print("\n✓ Top 15 collaborators identified and saved!")
print("✓ Ready for binary feature creation!")



Top 15 Most Frequent Collaborators:
 collaborator  frequency
     J Balvin       5352
    Bad Bunny       4778
        Ozuna       3623
       Khalid       2854
 Daddy Yankee       2811
     Anuel AA       2664
     Swae Lee       2411
      Farruko       2225
     Dua Lipa       1894
Lenny Tavárez       1818
        Quavo       1664
    Nicky Jam       1558
    Daft Punk       1390
  Nicki Minaj       1349
         Sech       1341

Top 15 Collaborators List (for binary features):
 1. J Balvin                       (Frequency: 5,352)
 2. Bad Bunny                      (Frequency: 4,778)
 3. Ozuna                          (Frequency: 3,623)
 4. Khalid                         (Frequency: 2,854)
 5. Daddy Yankee                   (Frequency: 2,811)
 6. Anuel AA                       (Frequency: 2,664)
 7. Swae Lee                       (Frequency: 2,411)
 8. Farruko                        (Frequency: 2,225)
 9. Dua Lipa                       (Frequency: 1,894)
10. Lenny Tavárez          

2D: Calculate Artist Popularity Scores

In [38]:


# Step 1: Add popularity_score column to artist_appearances_df
# Popularity score = 201 - rank (higher rank = lower number = higher popularity)
artist_appearances_df['popularity_score'] = 201 - artist_appearances_df['rank']

print("\nSample data with popularity scores (First 20 rows):")
print("="*80)
print(artist_appearances_df[['artist_name', 'song_id', 'rank', 'popularity_score']].head(20).to_string(index=False))

# Step 2: Calculate average popularity per artist
artist_popularity = artist_appearances_df.groupby('artist_name')['popularity_score'].mean().reset_index()
artist_popularity.columns = ['artist_name', 'avg_popularity']

# Round to 2 decimal places
artist_popularity['avg_popularity'] = artist_popularity['avg_popularity'].round(2)

# Sort by popularity descending
artist_popularity = artist_popularity.sort_values('avg_popularity', ascending=False)

print("\n" + "="*80)
print("Artist Popularity Statistics:")
print("="*80)
print(f"Total artists: {len(artist_popularity):,}")
print(f"\nPopularity score distribution:")
print(artist_popularity['avg_popularity'].describe())

print("\n" + "="*80)
print("Top 20 Most Popular Artists (by average popularity):")
print("="*80)
print(artist_popularity.head(20).to_string(index=False))

print("\n" + "="*80)
print("Examples from Spec:")
print("="*80)

example_artists = ['Bad Bunny', 'Peso Pluma', 'J Balvin']

for artist in example_artists:
    if artist in artist_popularity['artist_name'].values:
        pop = artist_popularity[artist_popularity['artist_name'] == artist].iloc[0]['avg_popularity']
        print(f"\n{artist}:")
        print(f"  - Average popularity score: {pop}")
    else:
        print(f"\n{artist}: Not found in dataset")

# Step 3: Convert to dictionary for fast lookup
artist_popularity_lookup = dict(zip(artist_popularity['artist_name'], artist_popularity['avg_popularity']))

print("\n" + "="*80)
print("Dictionary Conversion:")
print("="*80)
print(f"Total artists in lookup dictionary: {len(artist_popularity_lookup):,}")
print(f"Type: {type(artist_popularity_lookup)}")

# Show sample entries
print("\n" + "="*80)
print("Sample Lookup Entries (Top 10):")
print("="*80)
for i, (artist, pop) in enumerate(list(artist_popularity_lookup.items())[:10], 1):
    print(f"{i:2d}. '{artist}': {pop}")

# Verify lookup works
print("\n" + "="*80)
print("Testing Fast Lookup:")
print("="*80)
test_artists = ['Bad Bunny', 'Peso Pluma', 'J Balvin']
for artist in test_artists:
    if artist in artist_popularity_lookup:
        print(f"{artist}: {artist_popularity_lookup[artist]}")
    else:
        print(f"{artist}: Not found")

print("\n" + "="*80)
print("✓ Artist popularity scores calculated!")
print("✓ Dictionary ready for feature engineering!")
print("="*80)


Sample data with popularity scores (First 20 rows):
artist_name                song_id  rank  popularity_score
      Tyler 7KA4W4McWYRpgf0fWsJZWB    18               183
      Tyler 7KA4W4McWYRpgf0fWsJZWB    19               182
      Tyler 7KA4W4McWYRpgf0fWsJZWB    24               177
      Tyler 7KA4W4McWYRpgf0fWsJZWB    27               174
      Tyler 7KA4W4McWYRpgf0fWsJZWB    20               181
      Tyler 7KA4W4McWYRpgf0fWsJZWB    18               183
      Tyler 7KA4W4McWYRpgf0fWsJZWB    17               184
      Tyler 7KA4W4McWYRpgf0fWsJZWB    14               187
      Tyler 7KA4W4McWYRpgf0fWsJZWB    16               185
      Tyler 7KA4W4McWYRpgf0fWsJZWB    24               177
      Tyler 7KA4W4McWYRpgf0fWsJZWB    23               178
      Tyler 7KA4W4McWYRpgf0fWsJZWB    18               183
      Tyler 7KA4W4McWYRpgf0fWsJZWB    18               183
      Tyler 7KA4W4McWYRpgf0fWsJZWB    15               186
      Tyler 7KA4W4McWYRpgf0fWsJZWB    13               188
   

2E: Calculate Global Mean Popularity

In [39]:
global_mean_popularity = artist_popularity['avg_popularity'].mean()

print(f"\nGlobal Mean Popularity: {global_mean_popularity:.2f}")

print("\n" + "="*80)
print("Context:")
print("="*80)
print(f"Total artists in calculation: {len(artist_popularity):,}")
print(f"Min artist popularity: {artist_popularity['avg_popularity'].min():.2f}")
print(f"Max artist popularity: {artist_popularity['avg_popularity'].max():.2f}")
print(f"Median artist popularity: {artist_popularity['avg_popularity'].median():.2f}")
print(f"Global mean popularity: {global_mean_popularity:.2f}")

print("\n" + "="*80)
print("Use Case:")
print("="*80)
print("This global mean will be used as a fallback value when:")
print("  - A collaborator is unknown (not in training data)")
print("  - We need to calculate collaboration features")
print("  - Instead of using 0 or None, we use this average")

print("\n" + "="*80)
print("Example:")
print("="*80)
print("If a song has collaborator 'Unknown Artist':")
print(f"  - Lookup fails: 'Unknown Artist' not in artist_popularity_lookup")
print(f"  - Use fallback: global_mean_popularity = {global_mean_popularity:.2f}")

print("\n✓ Global mean popularity calculated!")
print(f"✓ Value saved in variable: global_mean_popularity = {global_mean_popularity:.2f}")
print("="*80)


Global Mean Popularity: 67.67

Context:
Total artists in calculation: 1,469
Min artist popularity: 1.00
Max artist popularity: 191.59
Median artist popularity: 67.69
Global mean popularity: 67.67

Use Case:
This global mean will be used as a fallback value when:
  - A collaborator is unknown (not in training data)
  - We need to calculate collaboration features
  - Instead of using 0 or None, we use this average

Example:
If a song has collaborator 'Unknown Artist':
  - Lookup fails: 'Unknown Artist' not in artist_popularity_lookup
  - Use fallback: global_mean_popularity = 67.67

✓ Global mean popularity calculated!
✓ Value saved in variable: global_mean_popularity = 67.67


In [40]:
print("="*80)
print("SUMMARY: CALCULATED FEATURES FROM TRAINING DATA")
print("="*80)

print("\n" + "="*80)
print("✅ 1. artist_stats_lookup (Dictionary)")
print("="*80)
print(f"Type: {type(artist_stats_lookup)}")
print(f"Size: {len(artist_stats_lookup):,} artists")
print("\nStructure:")
print("  {")
print("    'artist_name': {")
print("      'song_count': int,")
print("      'avg_rank': float,")
print("      'best_rank': int")
print("    },")
print("    ...")
print("  }")
print("\nUsage: Apply to aggregated train data for artist features")
print("\nExample:")
if 'Bad Bunny' in artist_stats_lookup:
    print(f"  artist_stats_lookup['Bad Bunny'] = {artist_stats_lookup['Bad Bunny']}")

print("\n" + "="*80)
print("✅ 2. top_15_collabs_list (List)")
print("="*80)
print(f"Type: {type(top_15_collabs_list)}")
print(f"Size: {len(top_15_collabs_list)} collaborators")
print("\nUsage: Create 15 binary collaboration features")
print("\nList:")
for i, collab in enumerate(top_15_collabs_list, 1):
    print(f"  {i:2d}. {collab}")

print("\n" + "="*80)
print("✅ 3. artist_popularity_lookup (Dictionary)")
print("="*80)
print(f"Type: {type(artist_popularity_lookup)}")
print(f"Size: {len(artist_popularity_lookup):,} artists")
print("\nStructure:")
print("  {")
print("    'artist_name': average_popularity_score,")
print("    ...")
print("  }")
print("\nUsage: Calculate numeric collaboration features")
print("\nExample:")
if 'Bad Bunny' in artist_popularity_lookup:
    print(f"  artist_popularity_lookup['Bad Bunny'] = {artist_popularity_lookup['Bad Bunny']}")

print("\n" + "="*80)
print("✅ 4. global_mean_popularity (Single Number)")
print("="*80)
print(f"Type: {type(global_mean_popularity)}")
print(f"Value: {global_mean_popularity:.2f}")
print("\nUsage: Fallback for unknown collaborators")
print("\nExample:")
print("  If collaborator not in artist_popularity_lookup:")
print(f"    Use fallback value: {global_mean_popularity:.2f}")

print("\n" + "="*80)
print("READY FOR FEATURE ENGINEERING!")
print("="*80)
print("\nNext Steps:")
print("  1. Aggregate daily data to song-level")
print("  2. Apply artist_stats_lookup to get artist features")
print("  3. Create 15 binary features using top_15_collabs_list")
print("  4. Calculate numeric collab features using artist_popularity_lookup")
print("  5. Use global_mean_popularity for unknown collaborators")
print("\n" + "="*80)

SUMMARY: CALCULATED FEATURES FROM TRAINING DATA

✅ 1. artist_stats_lookup (Dictionary)
Type: <class 'dict'>
Size: 1,030 artists

Structure:
  {
    'artist_name': {
      'song_count': int,
      'avg_rank': float,
      'best_rank': int
    },
    ...
  }

Usage: Apply to aggregated train data for artist features

Example:
  artist_stats_lookup['Bad Bunny'] = {'song_count': 63, 'avg_rank': 83.85, 'best_rank': 1}

✅ 2. top_15_collabs_list (List)
Type: <class 'list'>
Size: 15 collaborators

Usage: Create 15 binary collaboration features

List:
   1. J Balvin
   2. Bad Bunny
   3. Ozuna
   4. Khalid
   5. Daddy Yankee
   6. Anuel AA
   7. Swae Lee
   8. Farruko
   9. Dua Lipa
  10. Lenny Tavárez
  11. Quavo
  12. Nicky Jam
  13. Daft Punk
  14. Nicki Minaj
  15. Sech

✅ 3. artist_popularity_lookup (Dictionary)
Type: <class 'dict'>
Size: 1,469 artists

Structure:
  {
    'artist_name': average_popularity_score,
    ...
  }

Usage: Calculate numeric collaboration features

Example:
  artis

In [41]:

# Save all the lookup dictionaries and lists
features_to_save = {
    'artist_stats_lookup': artist_stats_lookup,
    'top_15_collabs_list': top_15_collabs_list,
    'artist_popularity_lookup': artist_popularity_lookup,
    'global_mean_popularity': global_mean_popularity
}

# Save to pickle file
with open('artist_features.pkl', 'wb') as f:
    pickle.dump(features_to_save, f)

print("✓ All features saved to 'artist_features.pkl'")
print(f"File contains: {list(features_to_save.keys())}")

✓ All features saved to 'artist_features.pkl'
File contains: ['artist_stats_lookup', 'top_15_collabs_list', 'artist_popularity_lookup', 'global_mean_popularity']


## For Combine Train + Val to Test